# Understanding Audio Features through Sonification

In this exercise notebook, we will segment, feature extract, and analyze audio files. Goals:

1. Detect onsets in an audio signal.
2. Segment the audio signal at each onset.
3. Compute features for each segment.
4. Gain intuition into the features by listening to each segment separately.

In [ ]:
import IPython.display as ipd
import matplotlib.pyplot as plt
import librosa.display
import numpy

from mirdotcom import mirdotcom

mirdotcom.init()

## Step 1: Retrieve Audio

Load the audio file `simple_loop.wav` into an array.

In [ ]:
fp = mirdotcom.get_audio("simple_loop.wav")
x, sr = librosa.load(fp)

Show the sample rate:

In [ ]:
print(sr)

Listen to the audio signal.

In [ ]:
ipd.Audio(x, rate=sr)

Display the audio signal.

In [ ]:
librosa.display.waveshow(x, sr=sr)
plt.ylabel("Amplitude")

Compute the short-time Fourier transform:

In [ ]:
X = librosa.stft(x)

For display purposes, compute the log amplitude of the STFT:

In [ ]:
Xmag = librosa.amplitude_to_db(X)

Display the spectrogram.

In [ ]:
# Play with the parameters, including x_axis and y_axis
librosa.display.specshow(Xmag, sr=sr, x_axis="time", y_axis="log")

## Step 2: Detect Onsets

Find the times, in seconds, when onsets occur in the audio signal.

In [ ]:
onset_frames = librosa.onset.onset_detect(y=x, sr=sr)
print(onset_frames)

In [ ]:
onset_times = librosa.frames_to_time(onset_frames, sr=sr)
print(onset_times)

Convert the onset frames into sample indices.

In [ ]:
onset_samples = librosa.frames_to_samples(onset_frames)
print(onset_samples)

Play a "beep" at each onset.

In [ ]:
# Use the `length` parameter so the click track is the same length as the original signal
clicks = librosa.clicks(times=onset_times, length=len(x))

In [ ]:
# Play the click track "added to" the original signal
ipd.Audio(x + clicks, rate=sr)

## Step 3: Segment the Audio

Save into an array, `segments`, 100-ms segments beginning at each onset.

In [ ]:
frame_sz = int(0.100 * sr)
segments = numpy.array([x[i : i + frame_sz] for i in onset_samples])

Here is a function that adds 300 ms of silence onto the end of each segment and concatenates them into one signal.

Later, we will use this function to listen to each segment, perhaps sorted in a different order.

In [ ]:
def concatenate_segments(segments, sr=22050, pad_time=0.300):
    padded_segments = [
        numpy.concatenate([segment, numpy.zeros(int(pad_time * sr))])
        for segment in segments
    ]
    return numpy.concatenate(padded_segments)

In [ ]:
concatenated_signal = concatenate_segments(segments, sr)

Listen to the newly concatenated signal.

In [ ]:
ipd.Audio(concatenated_signal, rate=sr)

## Step 4: Extract Features

For each segment, compute the zero crossing rate.

In [ ]:
zcrs = [sum(librosa.core.zero_crossings(segment)) for segment in segments]
print(zcrs)

Use `argsort` to find an index array, `ind`, such that `segments[ind]` is sorted by zero crossing rate.

In [ ]:
ind = numpy.argsort(zcrs)
print(ind)

Sort the segments by zero crossing rate, and concatenate the sorted segments.

In [ ]:
concatenated_signal = concatenate_segments(segments[ind], sr)

## Step 5: Listen to Segments

Listen to the sorted segments. What do you hear?

In [ ]:
ipd.Audio(concatenated_signal, rate=sr)

## More Exercises

Repeat the steps above using other features from [`librosa.feature`](https://librosa.org/doc/latest/feature.html), e.g. `rmse`, `spectral_centroid`, `spectral_bandwidth`.

Repeat the steps above for other audio files:

In [ ]:
mirdotcom.list_audio()